# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

In [40]:
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.ch06.array_stack import ArrayStack
from goodrich.exceptions import Empty

## 1. Registro de datos

Cada registro es una tupla de tres elementos:

```python
(sensor, variable, value)
```

Ejemplos:

```python
("S01", "temperature", 23.5)
("S02", "temperature", 25.1)
("S01", "humidity", 61.2)
```

Una combinación única `(sensor, variable)` identifica un dato dentro del estado actual.

## 2. Clase `DataProcessor`

Implemente:

```python
class DataProcessor:
    ...
```

Debe utilizar:

- un `ArrayQueue` para los registros pendientes;
- un `ArrayStack` para el historial de cambios;
- un `list` para mantener el estado actual.

### Restricción

No sustituya `ArrayQueue` o `ArrayStack` por `list`, `collections.deque` u otra estructura para realizar las funciones que corresponden a la Queue o al Stack.

La clase `DataProcessor` no debe imprimir resultados. Los métodos deben devolver los valores especificados. Las impresiones utilizadas para demostrar el funcionamiento deben realizarse en las celdas de prueba.

In [32]:
class DataProcessor:
    def __init__(self):
        # queue: registros pendientes por procesar
        self.pendientes = ArrayQueue()
        
        # stack: historial de cambios
        self.historial = ArrayStack()
        
        # lista: estado actual de los registros
        self.estado = []
    

## 3. `add(record)`

Agrega un registro a la `ArrayQueue`.

```python
processor.add(("S01", "temperature", 23.5))
```

Requisitos:

- agrega el registro a la cola;
- **no procesa** el registro;
- conserva el orden de llegada.

No se requiere un valor de retorno.

Debe rechazar registros que no tengan exactamente tres componentes o cuyo `value` no sea numérico. El tipo concreto de excepción para estos errores puede ser elegido por el estudiante, pero debe documentarse y utilizarse consistentemente.

In [33]:
def add(self, record):
        #  rwvisar la tupla tenga exactamente las 3 partes (sensor, variable, valor), si no error
        if len(record) != 3:
            raise ValueError("El registro debe tener exactamente 3 componentes,volver a intentar")
        
        # Luego verificamos que el tercer elemento  sea un numero
        if not isinstance(record[2], (int, float)):
            raise TypeError("El 3 valor debe ser numerico")
        
        # si pasa las dos pruebas se entra a la cola con el metodo enqueue
        self.pendientes.enqueue(record)

## 4. `process_next()`

Procesa el siguiente registro pendiente.

Debe:

1. obtener el siguiente registro de la Queue;
2. procesarlo;
3. actualizar el estado actual;
4. guardar en el Stack la información necesaria para poder deshacer exactamente ese cambio.

### FIFO

Si se ejecuta:

```python
add(A)
add(B)
add(C)
```

las llamadas sucesivas a `process_next()` deben devolver/procesar `A`, luego `B` y luego `C`.

### Actualización

Si se procesa:

```python
("S01", "temperature", 23.5)
```

en el estado se debe reflejar:

```python
('S01', 'temperature', 23.5)
```

Si después se procesa `("S01", "temperature", 27.0)`, el valor actual debe ser `27.0`.

### Historial

Se debe agregar al `Stack` respectivo

### Retorno

Debe devolver el registro que acaba de ser procesado.

### Queue vacía

Si no hay registros pendientes, debe producir `Empty, el error creado en el repositorio `Goodrich`

In [34]:
def process_next(self):
        #si la cola esta vacia, tirar error
        if self.pendientes.is_empty(): 
            raise Empty("No hay registros pendientes en la Queue")
            
        # Sacamos el primer registro de la fila, y lo dividimos para mayor facilidad de entendimiento
        registro = self.pendientes.dequeue()
        sensor, variable, valor_nuevo = registro
        
        # usamos esta variable para saber si encontramos el dato en la lista 
        dato_existe = False
        
        #Buscar en el estado actual(lista) y preparar el Stack, nuestro historial
        for i in range(len(self.estado)):
            # Revisamos si ya tenemos guardado ese sensor y esa variable
            if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                # Se encontro, entonces tenemos que guardar el valor viejo para despues
                valor_viejo = self.estado[i][2]
            
                # Le decimos que fue una actualización y cual era el valor anterior
                self.historial.push(( 'actualizar',sensor, variable, valor_viejo))
                
                # Actualizamos el estado actual(lista) con el valor nuevo
                self.estado[i] = (sensor, variable, valor_nuevo)
                dato_existe = True
                break
                
        # Si terminamos de revisar la lista y el dato no existia, lo metemos y ya, no actualizamos nada
        if not dato_existe:
            # Metemos al Stack la instrucción de que se acaba de crear algo nuevo, None por que no existia uno antes
            self.historial.push(('crear', sensor, variable, None))
            
            # Lo agregamos como nuevo a la lista del estado actual
            self.estado.append(registro)
            
        # a lo ultimo devolvemos el registro completo que acabamos de procesar
        return registro

## 5. `undo()`

Deshace el último cambio realizado mediante `process_next()`.

Ejemplo:

```text
20 → 25 → 30
```

Después de un `undo()`:

```text
20 → 25
```

Después de otro:

```text
20
```

Los cambios deben deshacerse en orden LIFO.

### Dato creado por primera vez

Suponga que inicialmente no existe `('S01', 'temperature', x)`, después de procesar:

```python
("S01", "temperature", 23.5)
```

el dato existe. Si se ejecuta `undo()`, debe volver a **no existir**.

### Historial vacío

Si no hay cambios que deshacer, debe producir `Empty`.

No se requiere un valor de retorno.

In [43]:
def undo(self):
        #Si el historial esta vacio, lanzamos el error
        if self.historial.is_empty():
            raise Empty("No hay cambios en el historial para deshacer")

        # Sacamos la ultima instrucción que guardamos en el historial(stack)
        instruccion = self.historial.pop()
        
        # separamos la informacion para que se facilite mas las cosas
        accion, sensor, variable, valor_viejo = instruccion

        # Revisamos que accion era para saber como echarla pa atras
        if accion == 'crear':
            # Si el dato se habia creado por primera vez, tenemos qeu eliminarlo de la lista :(
            for i in range(len(self.estado)):
                if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                    self.estado.pop(i) # pop(i) en una lista elimina el elemento en esa posición
                    break
                    
        elif accion == 'actualizar':
            # Si el dato se habia actualizado, tenemos que volver al valor viejo
            for i in range(len(self.estado)):
                if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                    self.estado[i] = (sensor, variable, valor_viejo)
                    break

## 6. `pending()`

Devuelve el número de registros que todavía esperan ser procesados.

Por ejemplo, después de:

```python
add(A)
add(B)
add(C)
```

`pending()` debe devolver `3`. Después de `process_next()`, debe devolver `2`.

In [44]:
def pending(self):
        # es el tamaño actual de la cola, de nuestros pendientes 
        return len(self.pendientes)

## 7. `current_value(sensor, variable)`

Devuelve el valor actual asociado con una combinación de sensor y variable.

Ejemplo:

```python
current_value("S01", "temperature")
```

puede devolver `23.5`.

Si nunca se ha procesado un registro para esa combinación, debe producir `KeyError`.

In [45]:
def current_value(self, sensor, variable):
        # eecorrer cada registro de nuestra lista de estado actual
        for i in range(len(self.estado)):
            # Comparamos si el sensor y la variable de cada tupla de la lsita es igual a la que comparamos
            if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                # si hay coincidencia, devolvemos el valor, el tercero de la tupla
                return self.estado[i][2]
                
        # Si el forr termina de dar todas las vueltas y no encontro nada, lanzamos el error
        raise KeyError(f"Nunca se ha procesado un registro para {sensor} y {variable}")

# 8. Ejemplo completo

Considere:

```python
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
```

Después de `add(A)`, `add(B)`, `add(C)`, la Queue contiene `A → B → C`.

Después de procesar A, el estado contiene:

```text
S01 / temperature → 20
```

Después de procesar B:

```text
S01 / temperature → 25
```

Después de procesar C:

```text
S01 / temperature → 25
S01 / humidity    → 60
```

Un `undo()` elimina el efecto de C. Otro `undo()` elimina el efecto de B. Otro `undo()` elimina el efecto de A y el estado vuelve a estar vacío.

In [50]:
#CODIGO COMPLETO :)
class DataProcessor:
    def __init__(self):
        self.pendientes = ArrayQueue()
        self.historial = ArrayStack()
        self.estado = []
        
    def add(self, record):
        if len(record) != 3:
            raise ValueError("El registro debe tener exactamente 3 componentes.")
        
        if not isinstance(record[2], (int, float)):
            raise TypeError("El valor debe ser numérico.")
        
        self.pendientes.enqueue(record)
        
    def process_next(self):
        if self.pendientes.is_empty(): 
            raise Empty("No hay registros pendientes en la Queue")
            
        registro = self.pendientes.dequeue()
        sensor, variable, valor_nuevo = registro
        
        dato_existe = False
        
        for i in range(len(self.estado)):
            if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                valor_viejo = self.estado[i][2]
                
                self.historial.push(('actualizar', sensor, variable, valor_viejo))
                
                self.estado[i] = (sensor, variable, valor_nuevo)
                dato_existe = True
                break
                
        if not dato_existe:
            self.historial.push(('crear', sensor, variable, None))
            self.estado.append(registro)
            
        return registro
        
    def undo(self):
        if self.historial.is_empty():
            raise Empty("No hay cambios en el historial para deshacer")

        instruccion = self.historial.pop()
        accion, sensor, variable, valor_viejo = instruccion

        if accion == 'crear':
            for i in range(len(self.estado)):
                if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                    self.estado.pop(i)
                    break
                    
        elif accion == 'actualizar':
            for i in range(len(self.estado)):
                if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                    self.estado[i] = (sensor, variable, valor_viejo)
                    break
                    
    def pending(self):
        return len(self.pendientes)
        
    def current_value(self, sensor, variable):
        for i in range(len(self.estado)):
            if self.estado[i][0] == sensor and self.estado[i][1] == variable:
                return self.estado[i][2]
                
        raise KeyError(f"Nunca se ha procesado un registro para {sensor} y {variable}")

# 9. Pruebas obligatorias

Incluya pruebas para, como mínimo:

1. `pending()` sobre un procesador vacío.
2. Agregar un registro.
3. Agregar varios registros.
4. Verificar procesamiento FIFO.
5. Procesar un registro.
6. Procesar varios registros.
7. Actualizar una variable existente.
8. Consultar el valor actual.
9. Realizar un `undo()`.
10. Realizar varios `undo()` consecutivos.
11. Procesar cuando la Queue está vacía.
12. Hacer `undo()` cuando el historial está vacío.
13. Deshacer la creación de un dato que antes no existía.
14. Hacer varios cambios sobre la misma variable.
15. Agregar un registro con formato incorrecto.
16. Agregar un registro cuyo valor no sea numérico.
17. Consultar un sensor/variable que nunca haya sido procesado.

In [55]:
print("--- 1. pending() sobre un procesador vacío ---")
p = DataProcessor()
print("Registros pendientes:", p.pending())

print("\n--- 2 y 3. Agregar un registro y Agregar varios registros ---")
p.add(("S01", "temperature", 20.0))
p.add(("S02", "humidity", 50.0))
p.add(("S03", "wind", 15.5))
print("Registros pendientes después de agregar 3:", p.pending())

print("\n--- 4 y 5. Verificar procesamiento FIFO y Procesar un registro ---")
procesado_1 = p.process_next()
print("Primer registro procesado (debe ser S01):", procesado_1)

print("\n--- 6. Procesar varios registros ---")
p.process_next()
p.process_next()
print("Estado actual de la lista:", p.estado)

print("\n--- 7 y 8. Actualizar una variable existente y Consultar el valor actual ---")
p.add(("S01", "temperature", 28.5))
p.process_next()
print("Nuevo valor consultado de S01/temperature (debe ser 28.5):", p.current_value("S01", "temperature"))

print("\n--- 9. Realizar un undo() (deshacer actualización) ---")
p.undo()
print("Valor de S01/temperature tras el undo (debe volver a 20.0):", p.current_value("S01", "temperature"))

print("\n--- 10 y 13. Varios undo() consecutivos y Deshacer creación de un dato ---")
p.undo() 
p.undo() 
p.undo() 
print("Estado tras vaciar todo el historial (debe estar vacío):", p.estado)

print("\n--- 11. Procesar cuando la Queue está vacía ---")
try:
    p.process_next()
except Exception as e:
    print("Éxito. Error atrapado:", type(e).__name__, "-", e)

print("\n--- 12. Hacer undo() cuando el historial está vacío ---")
try:
    p.undo()
except Exception as e:
    print("Éxito. Error atrapado:", type(e).__name__, "-", e)



--- 1. pending() sobre un procesador vacío ---
Registros pendientes: 0

--- 2 y 3. Agregar un registro y Agregar varios registros ---
Registros pendientes después de agregar 3: 3

--- 4 y 5. Verificar procesamiento FIFO y Procesar un registro ---
Primer registro procesado (debe ser S01): ('S01', 'temperature', 20.0)

--- 6. Procesar varios registros ---
Estado actual de la lista: [('S01', 'temperature', 20.0), ('S02', 'humidity', 50.0), ('S03', 'wind', 15.5)]

--- 7 y 8. Actualizar una variable existente y Consultar el valor actual ---
Nuevo valor consultado de S01/temperature (debe ser 28.5): 28.5

--- 9. Realizar un undo() (deshacer actualización) ---
Valor de S01/temperature tras el undo (debe volver a 20.0): 20.0

--- 10 y 13. Varios undo() consecutivos y Deshacer creación de un dato ---
Estado tras vaciar todo el historial (debe estar vacío): []

--- 11. Procesar cuando la Queue está vacía ---
Éxito. Error atrapado: Empty - No hay registros pendientes en la Queue

--- 12. Hacer un

In [56]:


print("\n--- 14. Hacer varios cambios sobre la misma variable ---")
p.add(("S99", "pressure", 100))
p.add(("S99", "pressure", 150))
p.add(("S99", "pressure", 200))
p.process_next()
p.process_next()
p.process_next()
print("Valor final consultado de S99/pressure (debe ser 200):", p.current_value("S99", "pressure"))

print("\n--- 15. Agregar un registro con formato incorrecto ---")
try:
    p.add(("S01", "temperature")) 
except Exception as e:
    print("Éxito. Error atrapado:", type(e).__name__, "-", e)

print("\n--- 16. Agregar un registro cuyo valor no sea numérico ---")
try:
    p.add(("S01", "temperature", "veinte")) 
except Exception as e:
    print("Éxito. Error atrapado:", type(e).__name__, "-", e)

print("\n--- 17. Consultar un sensor/variable que nunca haya sido procesado ---")
try:
    p.current_value("S_NUEVO", "nada")
except Exception as e:
    print("Éxito. Error atrapado:", type(e).__name__, "-", e)


--- 14. Hacer varios cambios sobre la misma variable ---
Valor final consultado de S99/pressure (debe ser 200): 200

--- 15. Agregar un registro con formato incorrecto ---
Éxito. Error atrapado: ValueError - El registro debe tener exactamente 3 componentes.

--- 16. Agregar un registro cuyo valor no sea numérico ---
Éxito. Error atrapado: TypeError - El valor debe ser numérico.

--- 17. Consultar un sensor/variable que nunca haya sido procesado ---
Éxito. Error atrapado: KeyError - 'Nunca se ha procesado un registro para S_NUEVO y nada'


# 10. Análisis de complejidad

Explique la complejidad temporal de:

- `add`
- `process_next`
- `undo`
- `pending`
- `current_value`

Justifique qué operaciones determinan cada complejidad e indique qué estructuras auxiliares utiliza el sistema.

# 11. Restricciones

1. Utilice las clases `ArrayStack` y `ArrayQueue` proporcionadas.
2. No las reemplace por `list`, `deque` u otra estructura equivalente.
3. No modifique las implementaciones proporcionadas.
4. La implementación debe estar contenida en `DataProcessor`.
5. Incluya las pruebas solicitadas.
6. Explique brevemente sus decisiones de diseño.

# 12. Bonus — `redo()`

Como extensión opcional, implemente:

```python
redo()
```

Después de un `undo()`, el sistema debe poder volver a aplicar el cambio que acaba de deshacerse.

Por ejemplo:

```text
20 → 25 → 30
undo()  → 20 → 25
redo()  → 20 → 25 → 30
```

El estudiante debe explicar qué estructuras utiliza para implementar `redo()` y por qué. No se proporciona la estrategia de implementación.


# 13. Entrega

La entrega debe contener:

- implementación completa de `DataProcessor`;
- pruebas solicitadas;
- explicación breve de las decisiones de diseño;
- análisis de complejidad temporal;
- si realiza el bonus, implementación y explicación de `redo()`.